In [1]:
from orderHIPERCAM import Hipercam_setup, Photometry
import os


/home/sittipong/.conda/envs/hcam-env/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
input_run = {
    'bias_data': {
        'runs': [['run002', 10, 0]], 
        'rawdir': '/lustre/MSSP/sittipong/all/data/2023_11_07',
        'ccd': '1'
    },
    'bias_dark': None, 
    'bias_flat': None, # {
        # 'runs': [['run002', 2, 0]], 
        # 'ccd': '1', 
    #     # 'rawdir': '/lustre/MSSP/sittipong/all/data/2023_11_07'
    # },
    'dark_data': None,
    'dark_flat': None,
    'flat_data': {
        'runs': [['run009', 2, 57]], 
        'ccd': '1', 
        'rawdir': '/lustre/MSSP/sittipong/all/data/2023_11_07'
    },
    'data': {
        'runs': [
            # ['run010', 1, 30], 
            ['run014', 1, 11], 
            # ['run018', 1, 20],
            ['run022', 1, 20], ['run026', 1, 20], ['run030', 1, 20],
            ['run037', 1, 20], ['run041', 1, 15]
        ],
        'ccd': '3', 
        'rawdir': '/lustre/MSSP/sittipong/all/data/2023_11_07'
    }
}
save_directory = os.path.join('/lustre/MSSP/sittipong/buildmodule', 'hcam_reduction')
default_raw_directory = '/lustre/MSSP/sittipong/all/data/2023_11_07'

In [3]:
print(f"---  Initializing Pipeline ---")
pipeline = Hipercam_setup(
    source='ul', 
    input_run_ul=input_run, 
    ccd='1', 
    save_dir=save_directory, 
    raw_dir=default_raw_directory,
    diagnostics=False
)

---  Initializing Pipeline ---
Work with source : ul
#### Grabbing run002 from /lustre/MSSP/sittipong/all/data/2023_11_07/run002 ####
Written frame 10 to run002_010.hcm
Written frame 11 to run002_011.hcm
Written frame 12 to run002_012.hcm
Written frame 13 to run002_013.hcm
Written frame 14 to run002_014.hcm
Written frame 15 to run002_015.hcm
Written frame 16 to run002_016.hcm
Written frame 17 to run002_017.hcm
Written frame 18 to run002_018.hcm
Written frame 19 to run002_019.hcm
Written frame 20 to run002_020.hcm
Written frame 21 to run002_021.hcm
Written frame 22 to run002_022.hcm
Written frame 23 to run002_023.hcm
Written frame 24 to run002_024.hcm
Written frame 25 to run002_025.hcm
Written frame 26 to run002_026.hcm
Written frame 27 to run002_027.hcm
Written frame 28 to run002_028.hcm
Written frame 29 to run002_029.hcm
Written frame 30 to run002_030.hcm
Written frame 31 to run002_031.hcm
Written frame 32 to run002_032.hcm
Written frame 33 to run002_033.hcm
Written frame 34 to run002

/home/sittipong/.conda/envs/hcam-env/lib/python3.12/site-packages/hipercam/scripts/makebias.py:200: UserWarning: You should not include the first frame of a run when making a bias from readout modes which do not have clear enabled since the first frame is different from all others.
  warnings.warn(


Loaded 41 CCDs
Computing and adjusting their mean levels
Combining them (clipped mean) and storing the result
Rejected 66434 pixels = 0.573% of the total

Final result written to /lustre/MSSP/sittipong/buildmodule/hcam_reduction/bias/master_bias_data.hcm
makebias finished
Processing bias_data finished...

#### Grabbing run009 from /lustre/MSSP/sittipong/all/data/2023_11_07/run009 ####
Written frame 2 to run009_002.hcm
Written frame 3 to run009_003.hcm
Written frame 4 to run009_004.hcm
Written frame 5 to run009_005.hcm
Written frame 6 to run009_006.hcm
Written frame 7 to run009_007.hcm
Written frame 8 to run009_008.hcm
Written frame 9 to run009_009.hcm
Written frame 10 to run009_010.hcm
Written frame 11 to run009_011.hcm
Written frame 12 to run009_012.hcm
Written frame 13 to run009_013.hcm
Written frame 14 to run009_014.hcm
Written frame 15 to run009_015.hcm
Written frame 16 to run009_016.hcm
Written frame 17 to run009_017.hcm
Written frame 18 to run009_018.hcm
Written frame 19 to run00

In [ ]:
photo_reduction = Photometry(base_dir=save_directory, lis=pipeline.data['lis'], diagnostics=True)
print('Import photometry done')
# print(f"\n{BLUE}---  End of Process ---{RESET}")

R_SKY1=12 
R_SKY2=15        
photo_reduction.setaper(
    ccd_label='1', 
    SKIP_BRIGHTEST=15,
    # list_file=science_lis,  # <--- Pass the .lis file here
    SIGMA_THRESHOLD=7,
    frame=5,
    R_SKY1=R_SKY1,
    R_SKY2=R_SKY2, 
    MARGIN_LEFT=17, 
    MARGIN_RIGHT=17, 
    MARGIN_BOTTOM=10, 
    MARGIN_TOP=30,
    diagnostics=True
    
)


photo_reduction.genred()

photo_reduction.modify_red(
    fit_method = 'moffat',
    fit_height_min_ref = 5.0,
    fit_height_min_nrf = 3.0,
    fit_half_width = 25.0,
    search_half_width = 15.0,
    )

photo_reduction.modify_red(
    target_section="extraction",
    **{"1": f"variable normal 1.80 3.0 10.0 2.5 {R_SKY1} {R_SKY1} 3.0 {R_SKY2} {R_SKY2}"}
)
photo_reduction.modify_red(fit_max_shift=8, search_smooth_fft='yes')

photo_reduction.reduce(diagnostics=True)
# photo_reduction.reduce()          # produces all.log

photo_reduction.solvwcs(          # reads all.log → wcs_radec.csv
    ccd_label='1',
    # scale_low=0.3,
    # scale_high=0.9,
    ra_center=322.4377,
    dec_center=-4.4853,
    radius=0.5,
    astrometry_cache='/lustre/MSSP/sittipong/astrometry_cache',
)
